# torch-batteries overhead over vanilla PyTorch

This notebook measures how much wall-clock time `torch-batteries` add on top of an equivalent hand-written PyTorch training loop in a typical setup: a progress bar with running loss and accuracy, and background data-loading workers.
All loops train the same torchvision ResNet on the same synthetic batches with the same optimizer, so the difference is what each framework costs as it is normally used. PyTorch Lightning can optionally be measured alongside for reference.

## What this notebook measures

- **Epoch time**: one full training epoch, timed over several epochs after warm-up.
- **Per-batch time**: epoch time divided by the number of batches.
- **Single-batch call time**: a one-batch `train` call, which exposes the fixed per-call setup cost.

[Training guide](https://michalszc.github.io/torch-batteries/guides/training/) · [Events API](https://michalszc.github.io/torch-batteries/reference/events/)

## 0. Prerequisites

The next cell installs torch-batteries only when it is not already importable, so a local environment keeps its own install and a fresh Colab runtime gets the PyPI release. Set `INCLUDE_LIGHTNING = True` there to also install and benchmark PyTorch Lightning.

A CUDA GPU is strongly recommended: on Colab, choose *Runtime → Change runtime type → T4 GPU*. The defaults take a few minutes on a T4. On CPU, lower `IMAGE_SIZE`, `BATCH_SIZE`, and `BATCHES_PER_EPOCH` first. 

Nothing is downloaded: the data is random and the models are randomly initialized.

In [1]:
import importlib.util

INCLUDE_LIGHTNING = True  # also benchmark PyTorch Lightning

# Keep existing (e.g. local editable) installs; otherwise install from PyPI.
packages = []
if importlib.util.find_spec("torch_batteries") is None:
    packages.append("torch-batteries")
if INCLUDE_LIGHTNING and importlib.util.find_spec("lightning") is None:
    packages.append("lightning")
if packages:
    %pip install {" ".join(packages)}

## 1. Configuration

Every setting that affects the measurement is a constant in the next cell, so it is easy to see and change.

- `MODEL_NAMES` are torchvision constructors (`resnet18`, `resnet34`, `resnet50`, ...).
- `BATCHES_PER_EPOCH` sets the epoch length. Framework overhead is a per-batch cost, so a moderate epoch is enough to amortize per-call setup. Samples are recycled from a pool of `SAMPLE_POOL_SIZE` images, so longer epochs do not use more memory.
- `WARMUP_EPOCHS` absorb CUDA context creation, cuDNN autotuning, and allocator growth; they are not timed.
- `TIMED_EPOCHS` epochs are timed per loop, each as a separate one-epoch call. The loops are interleaved, and the one that runs first rotates, so GPU clock and thermal drift affect all of them equally.
- `NUM_WORKERS` background workers load batches while the GPU trains, as in a typical setup. They are persistent, so worker start-up is paid once rather than in every timed call.

In [ ]:
import logging
import time
import warnings

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, clear_output, display
from torch import nn
from torch.nn import functional as nn_functional
from torch.utils.data import DataLoader, Dataset
from torchvision import models
from tqdm import tqdm

import torch_batteries
from torch_batteries import Battery, Event, EventContext, StepOutput, charge
from torch_batteries.utils.device import get_device

# Models and data shape
MODEL_NAMES = ("resnet18",)
NUM_CLASSES = 10
IMAGE_SIZE = 224
BATCH_SIZE = 64
BATCHES_PER_EPOCH = 50
SAMPLE_POOL_SIZE = 256

# Timing protocol
WARMUP_EPOCHS = 1
TIMED_EPOCHS = 10
SINGLE_BATCH_REPEATS = 20

# Statistics
BOOTSTRAP_RESAMPLES = 10_000
CONFIDENCE_LEVEL = 0.95

# Training and loader settings shared by all loops
LEARNING_RATE = 0.01
MOMENTUM = 0.9
NUM_WORKERS = 2
CUDNN_BENCHMARK = True
SEED = 42

LOOP_LABELS = {
    "vanilla": "vanilla PyTorch",
    "battery": "torch-batteries",
    "lightning": "Lightning",
}

DEVICE = get_device("auto")  # or "cuda", "mps", "cpu"
PIN_MEMORY = DEVICE.type == "cuda"
torch.backends.cudnn.benchmark = CUDNN_BENCHMARK

print(f"torch-batteries: {torch_batteries.__version__}")
print(f"torch: {torch.__version__}")
if INCLUDE_LIGHTNING:
    import lightning as L
    from lightning.pytorch.callbacks import TQDMProgressBar

    logging.getLogger("lightning.pytorch").setLevel(logging.WARNING)
    warnings.filterwarnings("ignore", ".*does not have many workers.*")
    print(f"lightning: {L.__version__}")

print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(DEVICE)}")

torch-batteries: 1.0.0
torch: 2.9.1+cu128
lightning: 2.6.6
Device: cuda
GPU: NVIDIA GeForce GTX 1660 Ti


## 2. Synthetic data

A fixed pool of random images and labels is generated once, and the dataset indexes it modulo the pool size. All loops therefore read identical batches through an identical `DataLoader`, and there is no download, decoding, or augmentation to blur the comparison.

In [3]:
class SyntheticImages(Dataset):
    def __init__(self, length, pool_size, image_size, num_classes, seed):
        generator = torch.Generator().manual_seed(seed)
        self.length = length
        self.images = torch.randn(
            pool_size, 3, image_size, image_size, generator=generator
        )
        self.labels = torch.randint(0, num_classes, (pool_size,), generator=generator)

    def __len__(self):
        return self.length

    def __getitem__(self, index):
        index %= len(self.labels)
        return self.images[index], self.labels[index]


def make_loader(batches):
    dataset = SyntheticImages(
        batches * BATCH_SIZE, SAMPLE_POOL_SIZE, IMAGE_SIZE, NUM_CLASSES, SEED
    )
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        persistent_workers=NUM_WORKERS > 0,
        pin_memory=PIN_MEMORY,
    )


epoch_loader = make_loader(BATCHES_PER_EPOCH)
single_batch_loader = make_loader(1)
print(
    f"Epoch: {len(epoch_loader)} batches of "
    f"{BATCH_SIZE} × 3 × {IMAGE_SIZE} × {IMAGE_SIZE}"
)

Epoch: 50 batches of 64 × 3 × 224 × 224


## 3. The training loops

Each loop does the same bookkeeping in its own idiomatic way: a progress bar updated every batch with the running loss and accuracy.

- `vanilla_train_epoch` is a hand-written loop: move the batch to the device, clear gradients, forward, cross-entropy, backward, optimizer step, then update running totals and a `tqdm` bar.
- `BenchmarkResNet` wraps a torchvision ResNet for torch-batteries. Its charged `TRAIN_STEP` returns the loss, predictions, and targets, and `Battery` computes the `accuracy` metric and shows both in its progress bar (`verbose=1`).
- When Lightning is included, `LightningResNet` wraps the same network and logs `loss` and `acc` to the progress bar with `self.log`. Each timed call builds a fresh `Trainer(max_epochs=1)` with no logger, checkpointing, or model summary, since torch-batteries writes none of these by default. In notebooks Lightning would otherwise render its bar as an ipywidgets HTML widget, so `TextProgressBar` switches it to the same plain-text `tqdm` bar the other loops use. The optimizer is created once and reused, so its momentum state carries over between calls exactly as in the other loops.

The progress bars differ in refresh rate and formatting, so the results show each framework's cost as it is normally used, not pure dispatch overhead. `timed` synchronizes the device before starting and after finishing the clock, so queued GPU kernels are attributed to the run that launched them.

In [4]:
def batch_accuracy(predictions, targets):
    return (predictions.argmax(dim=1) == targets).float().mean()


class BenchmarkResNet(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.network = getattr(models, model_name)(num_classes=NUM_CLASSES)

    def forward(self, images):
        return self.network(images)

    @charge(Event.TRAIN_STEP)
    def training_step(self, context: EventContext):
        images, targets = context["batch"]
        logits = self(images)
        return StepOutput(
            loss=nn_functional.cross_entropy(logits, targets),
            predictions=logits,
            targets=targets,
        )


def make_optimizer(model):
    return torch.optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=MOMENTUM)


def vanilla_train_epoch(model, optimizer, loader):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    with tqdm(loader, desc=LOOP_LABELS["vanilla"]) as progress:
        for images, targets in progress:
            images = images.to(DEVICE)
            targets = targets.to(DEVICE)
            optimizer.zero_grad()
            logits = model(images)
            loss = nn_functional.cross_entropy(logits, targets)
            loss.backward()
            optimizer.step()
            batch_size = len(targets)
            total_loss += loss.item() * batch_size
            total_correct += (logits.argmax(dim=1) == targets).sum().item()
            total_samples += batch_size
            progress.set_postfix_str(
                f"Loss={total_loss / total_samples:.4f}, "
                f"Accuracy={total_correct / total_samples:.4f}"
            )
    return total_loss / total_samples


def battery_train_epoch(battery, loader):
    return battery.train(loader, epochs=1, verbose=1)


if INCLUDE_LIGHTNING:

    class LightningResNet(L.LightningModule):
        def __init__(self, model_name):
            super().__init__()
            self.network = getattr(models, model_name)(num_classes=NUM_CLASSES)
            self.optimizer = make_optimizer(self)

        def forward(self, images):
            return self.network(images)

        def training_step(self, batch, batch_idx):
            images, targets = batch
            logits = self(images)
            loss = nn_functional.cross_entropy(logits, targets)
            self.log("loss", loss, on_step=True, on_epoch=True, prog_bar=True)
            self.log(
                "acc",
                batch_accuracy(logits, targets),
                on_step=True,
                on_epoch=True,
                prog_bar=True,
            )
            return loss

        def configure_optimizers(self):
            return self.optimizer

    class TextProgressBar(TQDMProgressBar):
        """Render plain-text bars like the other loops instead of notebook widgets."""

        def init_train_tqdm(self):
            return tqdm(desc=self.train_description, leave=True)

    def lightning_train_epoch(module, loader):
        trainer = L.Trainer(
            accelerator=DEVICE.type,
            devices=1,
            max_epochs=1,
            logger=False,
            enable_checkpointing=False,
            enable_model_summary=False,
            callbacks=[TextProgressBar()],
        )
        trainer.fit(module, loader)


def synchronize():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize(DEVICE)
    elif DEVICE.type == "mps":
        torch.mps.synchronize()


def timed(run):
    synchronize()
    start = time.perf_counter()
    run()
    synchronize()
    return time.perf_counter() - start

## 4. Benchmark protocol

For each model, identically initialized copies are created: one trained by the vanilla loop, one owned by a `Battery`, and optionally one fitted by Lightning. After the warm-up epochs, each repeat times one epoch with every loop, rotating which goes first. The single-batch measurement then repeats a one-batch call many times, which shows the fixed cost of entering and leaving a framework's training call.

Because every timed epoch runs all loops back to back, each epoch gives a paired comparison with the vanilla loop under the same conditions. Overheads are therefore medians of the per-epoch differences rather than differences of medians. The confidence interval of the epoch overhead comes from a percentile bootstrap: the per-epoch differences are resampled with replacement `BOOTSTRAP_RESAMPLES` times, and the interval spans the central `CONFIDENCE_LEVEL` share of the resampled medians.

In [5]:
def interleaved(runs, repeats):
    samples = {name: [] for name in runs}
    names = list(runs)
    for repeat in range(repeats):
        shift = repeat % len(names)
        for name in names[shift:] + names[:shift]:
            samples[name].append(timed(runs[name]))
    return samples


def benchmark(model_name):
    torch.manual_seed(SEED)
    vanilla_model = BenchmarkResNet(model_name).to(DEVICE)
    battery_model = BenchmarkResNet(model_name)
    battery_model.load_state_dict(vanilla_model.state_dict())
    vanilla_optimizer = make_optimizer(vanilla_model)
    battery = Battery(
        battery_model,
        device=DEVICE,
        optimizer=make_optimizer(battery_model),
        metrics={"accuracy": batch_accuracy},
    )
    if INCLUDE_LIGHTNING:
        lightning_model = LightningResNet(model_name)
        lightning_model.load_state_dict(vanilla_model.state_dict())

    def runs_for(loader):
        runs = {
            "vanilla": lambda: vanilla_train_epoch(
                vanilla_model, vanilla_optimizer, loader
            ),
            "battery": lambda: battery_train_epoch(battery, loader),
        }
        if INCLUDE_LIGHTNING:
            runs["lightning"] = lambda: lightning_train_epoch(lightning_model, loader)
        return runs

    interleaved(runs_for(epoch_loader), WARMUP_EPOCHS)
    epoch_samples = interleaved(runs_for(epoch_loader), TIMED_EPOCHS)
    single_samples = interleaved(runs_for(single_batch_loader), SINGLE_BATCH_REPEATS)
    return epoch_samples, single_samples


def bootstrap_median_ci(values, rng):
    resamples = rng.choice(values, size=(BOOTSTRAP_RESAMPLES, len(values)))
    medians = np.median(resamples, axis=1)
    tail = 100 * (1 - CONFIDENCE_LEVEL) / 2
    return np.percentile(medians, [tail, 100 - tail])


def summarize(epoch_samples, single_samples):
    rng = np.random.default_rng(SEED)
    vanilla_epochs = np.array(epoch_samples["vanilla"])
    vanilla_singles = np.array(single_samples["vanilla"])
    confidence = f"{100 * CONFIDENCE_LEVEL:g}%"
    rows = {}
    for name in epoch_samples:
        epochs = np.array(epoch_samples[name])
        singles = np.array(single_samples[name])
        overheads = 100 * (epochs - vanilla_epochs) / vanilla_epochs
        low, high = (
            bootstrap_median_ci(overheads, rng) if name != "vanilla" else (0.0, 0.0)
        )
        rows[LOOP_LABELS[name]] = {
            "epoch median ms": 1000 * np.median(epochs),
            "epoch stdev ms": 1000 * np.std(epochs, ddof=1),
            "overhead ms/batch": 1000
            * np.median(epochs - vanilla_epochs)
            / BATCHES_PER_EPOCH,
            "overhead %": np.median(overheads),
            f"overhead % CI {confidence} low": low,
            f"overhead % CI {confidence} high": high,
            "slower epochs": f"{np.sum(epochs > vanilla_epochs)}/{len(epochs)}",
            "single-batch median ms": 1000 * np.median(singles),
            "per-call overhead ms": 1000 * np.median(singles - vanilla_singles),
        }
    return pd.DataFrame.from_dict(rows, orient="index").rename_axis("loop")

## 5. Run the benchmark

Each model takes `(WARMUP_EPOCHS + TIMED_EPOCHS)` epochs plus `SINGLE_BATCH_REPEATS` single-batch calls per loop: two loops by default, three with Lightning. Each model gets its own table, with one row per loop; overhead columns are relative to the vanilla PyTorch row. The progress bars of a model's runs are cleared once it finishes, leaving only the tables.

In [6]:
results = {}
epoch_samples_by_model = {}
for model_name in MODEL_NAMES:
    epoch_samples, single_samples = benchmark(model_name)
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    epoch_samples_by_model[model_name] = epoch_samples
    results[model_name] = summarize(epoch_samples, single_samples)
    clear_output(wait=True)
    for name, table in results.items():
        display(Markdown(f"### {name}"), table.round(2))

### resnet18

,epoch median ms,epoch stdev ms,overhead ms/batch,overhead %,overhead % CI 95% low,overhead % CI 95% high,slower epochs,single-batch median ms,per-call overhead ms
loop,,,,,,,,,
vanilla PyTorch,10778.57,521.15,0.00,0.00,0.00,0.00,0/25,351.23,0.00
torch-batteries,10796.45,646.84,0.31,0.14,-0.46,1.46,14/25,282.70,-5.46
Lightning,11164.66,739.39,7.38,3.45,2.80,5.81,22/25,759.31,414.51


## 6. Individual epoch times

Medians are robust to an occasional slow epoch, but the individual samples show whether the timings were stable. Overhead smaller than the spread between epochs cannot be told apart from noise.

In [7]:
for model_name, samples in epoch_samples_by_model.items():
    epoch_times = pd.DataFrame(
        {
            LOOP_LABELS[name]: [1000 * value for value in values]
            for name, values in samples.items()
        },
        index=pd.RangeIndex(1, TIMED_EPOCHS + 1, name="epoch"),
    )
    display(Markdown(f"### {model_name} (ms)"), epoch_times.round(1))

### resnet18 (ms)

,vanilla PyTorch,torch-batteries,Lightning
epoch,,,
1,12322.2,12099.1,11556.3
2,10738.2,10788.5,10936.5
3,10652.2,10602.9,11008.4
4,10564.4,10802.2,11108.3
5,10907.1,11782.0,12643.8
6,11853.1,12408.9,13128.1
7,11715.9,12328.5,13309.8
8,10778.6,12512.8,12704.7
9,10830.4,10726.4,11107.7


## 7. Reading the results

- **Epoch overhead %** is the headline number: how much longer a realistic epoch takes under `Battery`. For GPU-bound models it should be small, because the framework's Python work runs while the GPU is busy or is tiny compared with a ResNet step.
- **The confidence interval** shows whether that overhead is distinguishable from noise: an interval that contains zero means it is not. **Slower epochs** counts how many of the paired epochs were slower than the vanilla loop; about half is what noise alone would give.
- **Overhead ms/batch** is roughly independent of the model. It grows with callbacks, metrics, and progress output, not with model size, so it matters most for very small models or very fast steps.
- **Single-batch overhead** includes per-call setup such as result history, progress, and loader resolution. It is paid once per `train` call, not once per batch. For Lightning it is dominated by building the `Trainer` and running `fit` setup, which is a real cost for workflows that call `fit` many times.